# Wind Forecast Error Analysis

This notebook analyses the accuracy of wind generation forecasts published by Elexon for **January 2024**.

We compare day-ahead and intra-day forecasts against outturn (actual) half-hourly wind generation, computing error metrics across multiple forecast horizons (1 h to 48 h).

**Data sources (Elexon BMRS API v1)**
| Dataset | Description |
|---------|-------------|
| `FUELHH` (fuelType=WIND) | Half-hourly actual wind generation (MW) |
| `WINDFOR` | Wind generation forecasts with publish timestamps |

## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import requests
import warnings

warnings.filterwarnings("ignore")

# Professional styling
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
})

print("Libraries loaded successfully.")

## 2. Fetch Actual Wind Generation (FUELHH)

We retrieve half-hourly outturn wind generation for January 2024.

In [ ]:
ACTUALS_URL = (
    "https://data.elexon.co.uk/bmrs/api/v1/datasets/FUELHH/stream"
    "?fuelType=WIND&settlementDateFrom=2024-01-01&settlementDateTo=2024-01-31"
)

print("Fetching actuals from Elexon FUELHH ...")
resp_actuals = requests.get(ACTUALS_URL, timeout=120)
resp_actuals.raise_for_status()

actuals = pd.DataFrame(resp_actuals.json())
print(f"Rows received: {len(actuals):,}")
actuals.head()

In [ ]:
# Inspect columns and parse timestamps
print("Columns:", actuals.columns.tolist())
print()

# Normalise column names to snake_case for convenience
actuals.columns = actuals.columns.str.strip()

# The key columns are typically: startTime and generation
time_col = [c for c in actuals.columns if "start" in c.lower() and "time" in c.lower()]
gen_col = [c for c in actuals.columns if "generation" in c.lower() or "quantity" in c.lower()]

if time_col and gen_col:
    actuals = actuals.rename(columns={time_col[0]: "startTime", gen_col[0]: "actual_mw"})
else:
    print("Warning: Could not auto-detect columns. Printing head for manual inspection.")
    print(actuals.head())

actuals["startTime"] = pd.to_datetime(actuals["startTime"], utc=True)
actuals["actual_mw"] = pd.to_numeric(actuals["actual_mw"], errors="coerce")
actuals = actuals.dropna(subset=["actual_mw"])
actuals = actuals.sort_values("startTime").reset_index(drop=True)

print(f"\nActuals range: {actuals['startTime'].min()} to {actuals['startTime'].max()}")
print(f"Records: {len(actuals):,}")
actuals[["startTime", "actual_mw"]].describe()

## 3. Fetch Wind Forecasts (WINDFOR)

We pull forecasts starting from 30 Dec 2023 so that we have 48-hour-ahead forecasts available for the start of January.

In [ ]:
FORECASTS_URL = (
    "https://data.elexon.co.uk/bmrs/api/v1/datasets/WINDFOR/stream"
    "?publishDateTimeFrom=2023-12-30T00:00:00Z&publishDateTimeTo=2024-02-01T00:00:00Z"
)

print("Fetching forecasts from Elexon WINDFOR ...")
resp_forecasts = requests.get(FORECASTS_URL, timeout=120)
resp_forecasts.raise_for_status()

forecasts = pd.DataFrame(resp_forecasts.json())
print(f"Rows received: {len(forecasts):,}")
forecasts.head()

In [ ]:
# Normalise forecast columns
forecasts.columns = forecasts.columns.str.strip()
print("Columns:", forecasts.columns.tolist())

# Detect publish time, start time, and forecast generation columns
pub_col = [c for c in forecasts.columns if "publish" in c.lower()]
start_col = [c for c in forecasts.columns if "start" in c.lower() and "time" in c.lower()]
fc_gen_col = [c for c in forecasts.columns if "generation" in c.lower() or "quantity" in c.lower() or "level" in c.lower()]

if pub_col and start_col and fc_gen_col:
    forecasts = forecasts.rename(columns={
        pub_col[0]: "publishTime",
        start_col[0]: "startTime",
        fc_gen_col[0]: "forecast_mw",
    })
else:
    print("Warning: Could not auto-detect columns.")
    print(forecasts.head())

forecasts["publishTime"] = pd.to_datetime(forecasts["publishTime"], utc=True)
forecasts["startTime"] = pd.to_datetime(forecasts["startTime"], utc=True)
forecasts["forecast_mw"] = pd.to_numeric(forecasts["forecast_mw"], errors="coerce")
forecasts = forecasts.dropna(subset=["forecast_mw"]).sort_values(["startTime", "publishTime"]).reset_index(drop=True)

forecasts["lead_time_h"] = (forecasts["startTime"] - forecasts["publishTime"]).dt.total_seconds() / 3600

print(f"\nForecasts range: {forecasts['startTime'].min()} to {forecasts['startTime'].max()}")
print(f"Publish range:   {forecasts['publishTime'].min()} to {forecasts['publishTime'].max()}")
print(f"Records: {len(forecasts):,}")
print(f"Lead time (hours) - min: {forecasts['lead_time_h'].min():.1f}, max: {forecasts['lead_time_h'].max():.1f}")

## 4. Select Forecasts at Each Horizon

For each target settlement period and each horizon *h*, we pick the **latest** forecast whose lead time `(startTime - publishTime)` is at least *h* hours.  This simulates a trader or system operator who needs a view *h* hours before delivery.

In [ ]:
HORIZONS = [1, 2, 4, 8, 12, 24, 48]  # hours

def select_forecast_at_horizon(forecasts_df, actuals_df, horizon_h):
    """
    For each settlement period in actuals, select the latest forecast
    published at least `horizon_h` hours before the start time.
    """
    # Filter forecasts with sufficient lead time
    eligible = forecasts_df[forecasts_df["lead_time_h"] >= horizon_h].copy()
    
    # For each startTime keep the row with the latest publishTime
    # (i.e. the freshest forecast that still satisfies the horizon)
    idx = eligible.groupby("startTime")["publishTime"].idxmax()
    selected = eligible.loc[idx, ["startTime", "forecast_mw", "publishTime", "lead_time_h"]].copy()
    
    merged = actuals_df[["startTime", "actual_mw"]].merge(selected, on="startTime", how="inner")
    merged["error"] = merged["forecast_mw"] - merged["actual_mw"]
    merged["abs_error"] = merged["error"].abs()
    merged["pct_error"] = (merged["abs_error"] / merged["actual_mw"].replace(0, np.nan)) * 100
    merged["horizon"] = horizon_h
    return merged

all_errors = pd.concat(
    [select_forecast_at_horizon(forecasts, actuals, h) for h in HORIZONS],
    ignore_index=True,
)

print(f"Combined error table: {len(all_errors):,} rows")
all_errors.head(10)

## 5. Compute Error Metrics by Horizon

| Metric | Definition |
|--------|------------|
| **MAE** | Mean Absolute Error (MW) |
| **RMSE** | Root Mean Square Error (MW) |
| **MAPE** | Mean Absolute Percentage Error (%) |
| **Median Error** | Median of absolute errors (MW) |
| **p99 Error** | 99th percentile of absolute errors (MW) |
| **Bias (MSE)** | Mean Signed Error - positive = over-forecast (MW) |

In [ ]:
def compute_metrics(group):
    return pd.Series({
        "MAE": group["abs_error"].mean(),
        "RMSE": np.sqrt((group["error"] ** 2).mean()),
        "MAPE": group["pct_error"].dropna().mean(),
        "Median_Error": group["abs_error"].median(),
        "p99_Error": group["abs_error"].quantile(0.99),
        "Bias_MSE": group["error"].mean(),
        "n_obs": len(group),
    })

metrics = all_errors.groupby("horizon", group_keys=False).apply(compute_metrics).reset_index()
metrics = metrics.rename(columns={"horizon": "Horizon (h)"})

# Display nicely
display_cols = ["Horizon (h)", "MAE", "RMSE", "MAPE", "Median_Error", "p99_Error", "Bias_MSE", "n_obs"]
metrics[display_cols].round(1)

## 6. Visualisations

### 6.1 Error Metrics vs Forecast Horizon

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: MAE and RMSE (MW)
ax = axes[0]
ax.plot(metrics["Horizon (h)"], metrics["MAE"], marker="o", linewidth=2, label="MAE")
ax.plot(metrics["Horizon (h)"], metrics["RMSE"], marker="s", linewidth=2, label="RMSE")
ax.plot(metrics["Horizon (h)"], metrics["Median_Error"], marker="^", linewidth=2, label="Median AE")
ax.set_xlabel("Forecast Horizon (hours)")
ax.set_ylabel("Error (MW)")
ax.set_title("Forecast Error vs Horizon")
ax.legend()
ax.set_xticks(HORIZONS)

# Right: MAPE (%)
ax = axes[1]
ax.plot(metrics["Horizon (h)"], metrics["MAPE"], marker="D", linewidth=2, color="tab:red")
ax.set_xlabel("Forecast Horizon (hours)")
ax.set_ylabel("MAPE (%)")
ax.set_title("Mean Absolute Percentage Error vs Horizon")
ax.set_xticks(HORIZONS)

fig.suptitle("Wind Forecast Accuracy - January 2024", fontsize=15, y=1.02)
fig.tight_layout()
plt.show()

### 6.2 Error by Hour of Day (Heatmap)

Heatmap of mean absolute error by hour of day and forecast horizon.  This reveals whether forecasts are systematically worse at certain times (e.g., overnight ramps).

In [ ]:
all_errors["hour"] = all_errors["startTime"].dt.hour

pivot = all_errors.pivot_table(
    values="abs_error",
    index="hour",
    columns="horizon",
    aggfunc="mean",
)
pivot.columns = [f"{int(c)}h" for c in pivot.columns]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    pivot,
    annot=True,
    fmt=".0f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "MAE (MW)"},
    ax=ax,
)
ax.set_xlabel("Forecast Horizon")
ax.set_ylabel("Hour of Day (UTC)")
ax.set_title("Mean Absolute Forecast Error by Hour of Day & Horizon")
fig.tight_layout()
plt.show()

### 6.3 Error Distribution Histogram

Distribution of signed forecast errors for selected horizons.  A distribution centred on zero indicates an unbiased forecast.

In [ ]:
selected_horizons = [1, 4, 12, 24, 48]

fig, ax = plt.subplots(figsize=(12, 6))
for h in selected_horizons:
    subset = all_errors[all_errors["horizon"] == h]["error"]
    ax.hist(subset, bins=60, alpha=0.45, label=f"{h}h ahead", edgecolor="white", linewidth=0.3)

ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("Forecast Error (MW)  [positive = over-forecast]")
ax.set_ylabel("Frequency")
ax.set_title("Distribution of Signed Forecast Errors by Horizon")
ax.legend(title="Horizon")
fig.tight_layout()
plt.show()

### 6.4 Bias Analysis (Mean Signed Error by Horizon)

Positive bias means the forecast systematically **over-predicts** wind output; negative bias means under-prediction.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

colors = ["tab:green" if v >= 0 else "tab:red" for v in metrics["Bias_MSE"]]
bars = ax.bar(metrics["Horizon (h)"].astype(str), metrics["Bias_MSE"], color=colors, edgecolor="black", width=0.6)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xlabel("Forecast Horizon (hours)")
ax.set_ylabel("Mean Signed Error (MW)")
ax.set_title("Forecast Bias by Horizon - January 2024")

# Annotate bars
for bar, val in zip(bars, metrics["Bias_MSE"]):
    y_pos = bar.get_height() + (20 if val >= 0 else -40)
    ax.text(bar.get_x() + bar.get_width() / 2, y_pos, f"{val:+.0f} MW",
            ha="center", va="bottom" if val >= 0 else "top", fontsize=10, fontweight="bold")

fig.tight_layout()
plt.show()

## 7. Summary

Key findings from the January 2024 wind forecast error analysis:

1. **Error grows with horizon** -- as expected, shorter lead-time forecasts are more accurate.
2. **MAE and RMSE** provide complementary views: RMSE penalises large outlier errors more heavily.
3. **The heatmap** reveals whether certain hours of the day are harder to forecast (e.g., evening ramp).
4. **Bias** indicates whether Elexon wind forecasts tend to over- or under-predict systematically.
5. **p99 errors** highlight worst-case exposure relevant for reserve sizing and risk management.

In [ ]:
print("=" * 60)
print("  Forecast Error Summary -- January 2024 Wind")
print("=" * 60)
for _, row in metrics.iterrows():
    print(f"\n  Horizon: {int(row['Horizon (h)'])}h")
    print(f"    MAE:          {row['MAE']:>8.0f} MW")
    print(f"    RMSE:         {row['RMSE']:>8.0f} MW")
    print(f"    MAPE:         {row['MAPE']:>7.1f} %")
    print(f"    Median AE:    {row['Median_Error']:>8.0f} MW")
    print(f"    p99 AE:       {row['p99_Error']:>8.0f} MW")
    print(f"    Bias (MSE):   {row['Bias_MSE']:>+8.0f} MW")
    print(f"    Observations: {int(row['n_obs']):>8,}")
print("\n" + "=" * 60)